# Facial Landmark, EAR & MAR Extraction

Extracts 68-point facial landmarks, then computes **EAR** (Eye Aspect Ratio) and **MAR** (Mouth Aspect Ratio) per frame.

**Prerequisites:** Run the scene-detection notebook first to produce segmented videos.  
**Configure:** Set `base_dir` and `output_csv_path` in the cells below before running.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

# NOTE: This notebook was originally run in Google Colab.
# If using Colab, uncomment the line below to mount Google Drive:
# from google.colab import drive


In [ ]:
# --- Environment setup (configure before running) ---
# Google Colab only — uncomment to mount Drive:
# drive.mount('/content/drive')

In [ ]:
# --- Paths (update these for your environment) ---
# Input: directory of segmented videos (one subfolder per source video)
base_dir = 'data/segments'  # e.g. 'data/segments' or your local path

# Output: directory for optional SPIGA visualization exports
video_out_path_dft = 'data/output'


In [ ]:
# Clone SPIGA if not already present (run once from your project root)
# !git clone https://github.com/andresprados/SPIGA.git
# %cd SPIGA


In [ ]:
# Install SPIGA in editable mode (run from inside the cloned SPIGA directory)
# !pip install -e .


In [ ]:
# !pip install sort-track


In [ ]:
# !pip install retina-face


In [ ]:
# Ensure output path exists
if not os.path.exists(video_out_path_dft):
    os.makedirs(video_out_path_dft)

In [ ]:
from sort.tracker import SortTracker

In [ ]:
# Custom Tracker based on new RetinaSortTracker class
from spiga.demo.analyze.track.retinasort.config import cfg_retinasort
from spiga.demo.analyze.analyzer import VideoAnalyzer
from spiga.demo.visualize.viewer import Viewer
from spiga.demo.analyze.features.face import Face
from spiga.demo.analyze.extract.spiga_processor import SPIGAProcessor

In [ ]:
from spiga.demo.analyze.track.tracker import Tracker

In [ ]:
!pip install mtcnn


In [ ]:
import numpy as np
from mtcnn import MTCNN
import spiga.demo.analyze.track.tracker as tracker
import spiga.demo.analyze.features.face as ft_face

class MTCNNSortTracker(tracker.Tracker):

    def __init__(self):
        super().__init__()
        self.detector = MTCNN()
        self.obj_type = ft_face.Face
        self.attributes += ['bbox', 'face_id', 'key_landmarks']

    def process_frame(self, image, tracked_obj):
        features = self.detector.detect_faces(image)
        bboxes = []
        for face in features:
            x, y, width, height = face['box']
            bbox = [x, y, x + width, y + height, 0]  # Adjust the format as needed
            bboxes.append(bbox)
        final_tracked_obj = []
        for idx, bbox in enumerate(bboxes):
            founded_flag = False
            for past_obj in tracked_obj:
                if past_obj.face_id == bbox[-1]:
                    past_obj.bbox = bbox[:5]
                    past_obj.key_landmarks = features[idx].get('keypoints')
                    final_tracked_obj.append(past_obj)
                    tracked_obj.remove(past_obj)
                    founded_flag = True
                    break

            if not founded_flag:
                new_obj = self.obj_type()
                new_obj.bbox = bbox[:5]
                new_obj.face_id = bbox[4]
                new_obj.key_landmarks = features[idx].get('keypoints')
                final_tracked_obj.append(new_obj)

        return final_tracked_obj

    def plot_features(self, image, features, plotter, show_attributes):
        if 'bbox' in show_attributes:
            image = plotter.bbox.draw_bbox(image, features.bbox)
        if 'face_id' in show_attributes:
            text_id = 'Face Id: %i' % features.face_id
            image = plotter.bbox.draw_bbox_text(image, features.bbox, text_id, offset=(0, -10), color=plotter.basic.colors['blue'])
            image = plotter.bbox.draw_bbox_line(image, features.bbox)
        return image


In [ ]:
# Update video_app function to use custom MTCNNSortTracker
def video_app(input_name, spiga_dataset='300wpublic', fps=30, save=False,
              output_path=video_out_path_dft, video_shape=None, visualize=True, plot=('fps', 'face_id', 'landmarks', 'headpose')):

    processor = SPIGAProcessor(dataset=spiga_dataset)
    faces_tracker = MTCNNSortTracker()  # Use the updated MTCNNSortTracker

    all_landmarks = []

    try:
        capture = cv2.VideoCapture(input_name)
        video_name = input_name.split('/')[-1][:-4]  # Extract video name without extension
    except Exception as e:
        raise ValueError(f'Input video path {input_name} not valid: {e}')

    if capture is not None:
        if video_shape is not None:
            vid_w, vid_h = video_shape
        else:
            vid_w, vid_h = int(capture.get(3)), int(capture.get(4))
        viewer = Viewer('face_app', width=vid_w, height=vid_h, fps=fps)
        if save:
            viewer.record_video(output_path, video_name)

        faces_analyzer = VideoAnalyzer(faces_tracker, processor=processor)

        if visualize:
            viewer.start_view()
        while capture.isOpened():
            ret, frame = capture.read()
            if ret:
                faces_analyzer.process_frame(frame)
                landmarks = faces_analyzer.get_attributes('landmarks')
                if landmarks:
                    for landmark in landmarks:
                        all_landmarks.append({
                            'frame': int(capture.get(cv2.CAP_PROP_POS_FRAMES)),
                            'landmarks': landmark
                        })
                key = viewer.process_image(frame, drawers=[faces_analyzer], show_attributes=plot)
                if key:
                    break
            else:
                break

        capture.release()
        viewer.close()

    return all_landmarks


In [ ]:
def calculate_ear(landmarks):
    # Define the indices for the left and right eyes based on SPIGA output
    left_eye_indices = [36, 37, 38, 39, 40, 41]
    right_eye_indices = [42, 43, 44, 45, 46, 47]

    # Extract eye landmarks
    left_eye = np.array([landmarks[i] for i in left_eye_indices])
    right_eye = np.array([landmarks[i] for i in right_eye_indices])

    # Calculate EAR for left eye
    left_eye_height = np.linalg.norm(left_eye[1] - left_eye[5]) + np.linalg.norm(left_eye[2] - left_eye[4])
    left_eye_width = np.linalg.norm(left_eye[0] - left_eye[3])
    ear_left = left_eye_height / (2.0 * left_eye_width)

    # Calculate EAR for right eye
    right_eye_height = np.linalg.norm(right_eye[1] - right_eye[5]) + np.linalg.norm(right_eye[2] - right_eye[4])
    right_eye_width = np.linalg.norm(right_eye[0] - right_eye[3])
    ear_right = right_eye_height / (2.0 * right_eye_width)

    return (ear_left + ear_right) / 2.0  # Average EAR for both eyes

def calculate_mar(landmarks):
    # Define the indices for the mouth based on SPIGA output
    mouth_indices = [48, 51, 54, 57]  # Include lower middle lip

    # Extract mouth landmarks
    mouth = np.array([landmarks[i] for i in mouth_indices])

    # Calculate MAR
    mouth_height = np.linalg.norm(mouth[1] - mouth[3])  # Upper middle to lower middle
    mouth_width = np.linalg.norm(mouth[0] - mouth[2])  # Left corner to right corner
    mar = mouth_height / mouth_width if mouth_width != 0 else 0  # Avoid division by zero

    return mar

In [ ]:
# Initialize an empty list to store results
results = []

# Traverse all folders and videos in the directory
for folder_name in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder_name)
    if os.path.isdir(folder_path):
        for video_name in os.listdir(folder_path):
            video_path = os.path.join(folder_path, video_name)
            if video_path.endswith('.mp4') or video_path.endswith('.avi'):  # Assuming video files
                print(f"Processing video: {folder_name}/{video_name}")

                # Run the video app to extract landmarks
                landmarks_data = video_app(video_path, spiga_dataset='300wpublic', save=False, visualize=False)

                # For each frame's landmarks, calculate EAR and MAR
                for entry in landmarks_data:
                    landmarks = entry['landmarks']
                    ear = calculate_ear(landmarks)
                    mar = calculate_mar(landmarks)

                    # Append the data to the results list
                    results.append({
                        'video_name': f"{folder_name}/{video_name}",
                        'landmarks': landmarks,  # You may serialize landmarks as a string if needed
                        'EAR': ear,
                        'MAR': mar
                    })


In [ ]:
# Convert results to a DataFrame
df = pd.DataFrame(results)

In [ ]:
# Save DataFrame to CSV
output_csv_path = 'data/output/landmarks_ear_mar.csv'  # update path as needed
df.to_csv(output_csv_path, index=False)


In [ ]:

print(f"CSV file saved at {output_csv_path}")